## PydanticOutputParser
> LLM의 텍스트 출력을 Pydantic 모델(BaseModel)로 자동 변환해 주는 LangChain 출력 파서<br>
> LLM이 자연어로 응답한 텍스트를 사람이 직접 파싱X , 자동으로 엄격한 구조화된 데이터로 바꿔줌
### Pydantic
> python에서 데이터 검증(data validation)과 데이터 모델링(data modeling)을 간단하고 강력하게 할 수 있도록 도와주는 라이브러리<br>
> python 타입 힌트를 실제로 강제하고 검증까지 해주는 라이브러리
### Parser

In [1]:
from pydantic import BaseModel, Field
from typing import List

# 한국 음식 정보를 위한 pydantic 모델 정의
class KoreanFood(BaseModel):
    # 변수 명 : 변수 타입 = Field(설명, 제약조건)
    # Field() : 이 필드에 대한 메타데이터(설명, 제약조건 등)를 추가하는 도구
    name: str = Field(description="음식 이름")
    category: str = Field(description="음식 카테고리 (예: 밥류, 국물류, 반찬류 등)")
    ingredients: List[str] = Field(description="주요 재료 목록")
    region: str = Field(description="대표적인 지역 (예: 서울, 부산, 전라도 등)")
    taste: str = Field(description="맛의 특징 (예: 매운맛, 단맛, 짠맛 등)")
    description: str = Field(description="음식에 대한 간단한 설명")

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

# Pydantic 출력 파서 초기화
# LLM의 텍스트 응답을 KoreanFood 형태의 구조화된 데이터로 변환하기 위한 파서
food_parser = PydanticOutputParser(pydantic_object = KoreanFood)

In [ ]:
# 출력 형식 지침 가져오기
format_instructions = food_parser.get_format_instructions()

print("출력 형식 지침:")
# 파서에 대한 포멧 템플릿 형식 가져와서 출력
print(format_instructions)

출력 형식 지침:
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "음식 이름", "title": "Name", "type": "string"}, "category": {"description": "음식 카테고리 (예: 밥류, 국물류, 반찬류 등)", "title": "Category", "type": "string"}, "ingredients": {"description": "주요 재료 목록", "items": {"type": "string"}, "title": "Ingredients", "type": "array"}, "region": {"description": "대표적인 지역 (예: 서울, 부산, 전라도 등)", "title": "Region", "type": "string"}, "taste": {"description": "맛의 특징 (예: 매운맛, 단맛, 짠맛 등)", "title": "Taste", "type": "string"}, "description": {"description": "음식에 대한 간단한 설명", "title": 

## Prompt

In [ ]:
from langchain_core.prompts import PromptTemplate

# 한국 음식 정보 추출을 위한 프롬프트 템플릿
food_prompt = PromptTemplate(
    template="""
    다음 한국 음식에 대한 정보를 자세히 분석해서 구조화된 형태로 추출 해주세요.

    음식: {food_name}

    {format_instructions}
    """,
    # 템플릿에서 나중에 값으로 채워 넣을 입력 변수 이름
    input_variables=['food_name'],
    # 입력된 변수를 템플릿에 더해져서 모델에게 전달된 후 표현될 형식
    partial_variables = {"format_instructions":format_instructions}
)

In [ ]:
# 사용자가 직접 입력해야하는 변수
food_prompt.input_variables

['food_name']

In [ ]:
# 모델에게 전달할 규칙이나 명령에 대한 템플릿
print(food_prompt.template)


    다음 한국 음식에 대한 정보를 자세히 분석해서 구조화된 형태로 추출 해주세요.

    음식: {food_name}

    {format_instructions}
    


In [ ]:
#  아직 모델을 불러오지 못해서 그냥 템플릿만 가져옴
print(food_prompt.invoke({"food_name":"김치"}).text)


    다음 한국 음식에 대한 정보를 자세히 분석해서 구조화된 형태로 추출 해주세요.

    음식: 김치

    The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "음식 이름", "title": "Name", "type": "string"}, "category": {"description": "음식 카테고리 (예: 밥류, 국물류, 반찬류 등)", "title": "Category", "type": "string"}, "ingredients": {"description": "주요 재료 목록", "items": {"type": "string"}, "title": "Ingredients", "type": "array"}, "region": {"description": "대표적인 지역 (예: 서울, 부산, 전라도 등)", "title": "Region", "type": "string"}, "taste": {"description": "맛의 특징 (예: 매운맛, 단맛, 짠맛 등)", "title": "Taste", "type": "string"}, "

## Model
### OpenAI API Key 발

In [ ]:
from dotenv import load_dotenv

# dotenv 파일에서 환경변수 로드
load_dotenv()

In [ ]:
import os 

# API 키 확인
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("OpenAI API 키가 설정되었습니다. (GPT 모델 사용)")
else:
    print("OpenAI API 키가 없습니다.")

In [ ]:
from langchain_openai import ChatOpenAI
# chatgpt-5 이후 모델
model = ChatOpenAI(
    model="gpt-5-nano",             # 모델 명
    reasoning_effort="high",        # 논리성 강화, 추론에 더 많은 계산/노력을 사용하도록 설정
)

## Chain with Parser
> 사용자 입력 -> 프롬프트 구성 -> LLM 호출 -> 출력 파싱 -> 결과 반환

In [ ]:
# 체인 생성
food_chain = food_prompt | model | food_parser

In [ ]:
# 생성된 체인을 불러와보기
food_chain

In [9]:
# 김치에 대한 정보 추출
result = food_chain.invoke({"food_name":"김치"})

NameError: name 'food_chain' is not defined

In [ ]:
# 모델에 대해서 김치에 대한 정보를 원하는 템플릿으로 받아온 것을 출
result

In [ ]:
print("김치 정보:")
print(f"이름: {result.name}")
print(f"카테고리: {result.category}")
print(f"재료: {', '.join(result.ingredients)}")
print(f"지역: {result.region}")
print(f"맛: {result.taste}")
print(f"설명: {result.description}")